# MB5B — Centro 4128 — Datalake

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario`
**Domínio:** Estoque diario (saldo inicial, entradas, saidas, saldo final)
**Filtro do cenário:** `cod_centro = '4128'`
**Colunas:** 12 · **Clustering declarado:** _(nenhum)_

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## Objetivo

Extrair e caracterizar **toda** a base do centro 4128 para comparação com o extrato do SAP.

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados da tabela |
| 2 | Volumetria e representatividade do cenário |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| 6 | Preenchimento de todas as colunas |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Perfil numérico |
| **10** | **Totais para conciliação com o SAP** |
| 11 | Datas |
| 12 | Códigos e zeros à esquerda |
| **13** | **Chaves normalizadas para join** |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| 18 | Análises específicas |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados da tabela

In [ ]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario;

In [ ]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario;

In [ ]:
-- Ultimas gravacoes (falha se for view)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario LIMIT 20;

## 2. Volumetria e representatividade

Quanto o centro 4128 representa do total da tabela.

In [ ]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario)                                   AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128')                               AS linhas_centro_4128,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario), 4)                  AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro`) FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario)                     AS centros_na_tabela;

## 3. Confirmação do filtro

Confirma que o valor `4128` existe e que não há variação de formato
(espaços, zeros à esquerda) que faça o filtro perder linhas silenciosamente.

**Se retornar mais de uma linha, o filtro `= '4128'` está incompleto.**

In [ ]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro` AS STRING)                    AS valor_bruto,
       length(CAST(`cod_centro` AS STRING))            AS comprimento,
       COUNT(*)                                   AS linhas
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario
WHERE regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') = '4128'
   OR trim(CAST(`cod_centro` AS STRING)) = '4128'
GROUP BY CAST(`cod_centro` AS STRING), length(CAST(`cod_centro` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [ ]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'),
g AS (
  SELECT 'cod_empresa + cod_centro + cod_material + dt_estoque' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_centro + cod_material + dt_estoque' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_centro`, `cod_material`, `dt_estoque` FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_empresa + cod_centro + cod_material + dt_estoque + sg_unidade_medida' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, `sg_unidade_medida` FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `cod_empresa + cod_centro + cod_material + dt_estoque`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [ ]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
GROUP BY `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [ ]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
),
dup AS (
  SELECT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` FROM cen GROUP BY `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`cod_empresa` <=> dup.`cod_empresa` AND c.`cod_centro` <=> dup.`cod_centro` AND c.`cod_material` <=> dup.`cod_material` AND c.`dt_estoque` <=> dup.`dt_estoque`
),
agg AS (
  SELECT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `sg_unidade_medida`) AS `sg_unidade_medida`,
         COUNT(DISTINCT `estoque_inicial`) AS `estoque_inicial`,
         COUNT(DISTINCT `entrada`) AS `entrada`,
         COUNT(DISTINCT `saida`) AS `saida`,
         COUNT(DISTINCT `estoque_final`) AS `estoque_final`,
         COUNT(DISTINCT `dh_carga`) AS `dh_carga`,
         COUNT(DISTINCT `dh_atualizacao`) AS `dh_atualizacao`
  FROM d GROUP BY `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(8,
    'desc_material', MAX(`desc_material`),
    'sg_unidade_medida', MAX(`sg_unidade_medida`),
    'estoque_inicial', MAX(`estoque_inicial`),
    'entrada', MAX(`entrada`),
    'saida', MAX(`saida`),
    'estoque_final', MAX(`estoque_final`),
    'dh_carga', MAX(`dh_carga`),
    'dh_atualizacao', MAX(`dh_atualizacao`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4128 — ou o contrário.
Por isso a varredura é feita sobre o recorte, não sobre a base completa.

In [ ]:
-- 6. PREENCHIMENTO NO CENTRO 4128
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'),
perf AS (
  SELECT stack(12,
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'sg_unidade_medida', 'string', COUNT_IF(`sg_unidade_medida` IS NULL), COUNT_IF(`sg_unidade_medida` IS NOT NULL AND lower(trim(`sg_unidade_medida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida`) RLIKE '^0+([.,]0+)?$'),
    'dt_estoque', 'string', COUNT_IF(`dt_estoque` IS NULL), COUNT_IF(`dt_estoque` IS NOT NULL AND lower(trim(`dt_estoque`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_estoque`) RLIKE '^0+([.,]0+)?$'),
    'estoque_inicial', 'double', COUNT_IF(`estoque_inicial` IS NULL), 0L, COUNT_IF(`estoque_inicial` = 0),
    'entrada', 'double', COUNT_IF(`entrada` IS NULL), 0L, COUNT_IF(`entrada` = 0),
    'saida', 'double', COUNT_IF(`saida` IS NULL), 0L, COUNT_IF(`saida` = 0),
    'estoque_final', 'double', COUNT_IF(`estoque_final` IS NULL), 0L, COUNT_IF(`estoque_final` = 0),
    'dh_carga', 'timestamp', COUNT_IF(`dh_carga` IS NULL), 0L, 0L,
    'dh_atualizacao', 'timestamp', COUNT_IF(`dh_atualizacao` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                       THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                              AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [ ]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'),
card AS (
  SELECT stack(12,
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'sg_unidade_medida', 'string', approx_count_distinct(`sg_unidade_medida`),
    'dt_estoque', 'string', approx_count_distinct(`dt_estoque`),
    'estoque_inicial', 'double', approx_count_distinct(`estoque_inicial`),
    'entrada', 'double', approx_count_distinct(`entrada`),
    'saida', 'double', approx_count_distinct(`saida`),
    'estoque_final', 'double', approx_count_distinct(`estoque_final`),
    'dh_carga', 'timestamp', approx_count_distinct(`dh_carga`),
    'dh_atualizacao', 'timestamp', approx_count_distinct(`dh_atualizacao`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma, dentro do cenário.

In [ ]:
-- 8. DOMINIO DAS CATEGORICAS
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128' GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida' AS coluna, CAST(`sg_unidade_medida` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128' GROUP BY `sg_unidade_medida` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

Campos `double` exigem tolerância de 0,005 na comparação com o SAP.

In [ ]:
-- 9. PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(4,
    'estoque_inicial', 'double', COUNT(`estoque_inicial`), CAST(MIN(`estoque_inicial`) AS DOUBLE), CAST(MAX(`estoque_inicial`) AS DOUBLE), CAST(AVG(`estoque_inicial`) AS DOUBLE), CAST(percentile_approx(`estoque_inicial`, 0.5) AS DOUBLE), COUNT_IF(`estoque_inicial` < 0), COUNT_IF(`estoque_inicial` = 0),
    'entrada', 'double', COUNT(`entrada`), CAST(MIN(`entrada`) AS DOUBLE), CAST(MAX(`entrada`) AS DOUBLE), CAST(AVG(`entrada`) AS DOUBLE), CAST(percentile_approx(`entrada`, 0.5) AS DOUBLE), COUNT_IF(`entrada` < 0), COUNT_IF(`entrada` = 0),
    'saida', 'double', COUNT(`saida`), CAST(MIN(`saida`) AS DOUBLE), CAST(MAX(`saida`) AS DOUBLE), CAST(AVG(`saida`) AS DOUBLE), CAST(percentile_approx(`saida`, 0.5) AS DOUBLE), COUNT_IF(`saida` < 0), COUNT_IF(`saida` = 0),
    'estoque_final', 'double', COUNT(`estoque_final`), CAST(MIN(`estoque_final`) AS DOUBLE), CAST(MAX(`estoque_final`) AS DOUBLE), CAST(AVG(`estoque_final`) AS DOUBLE), CAST(percentile_approx(`estoque_final`, 0.5) AS DOUBLE), COUNT_IF(`estoque_final` < 0), COUNT_IF(`estoque_final` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 10. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel extraído do SAP e compare linha a linha.
Divergência de total é o teste mais rápido para detectar registro faltando ou duplicado —
e cobre o ponto cego da contagem de linhas, que sozinha não prova nada.

In [ ]:
-- 10. TOTAIS PARA CONCILIACAO
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(4,
    'estoque_inicial', CAST(SUM(`estoque_inicial`) AS DOUBLE), CAST(ROUND(SUM(`estoque_inicial`), 2) AS STRING), COUNT(`estoque_inicial`),
    'entrada', CAST(SUM(`entrada`) AS DOUBLE), CAST(ROUND(SUM(`entrada`), 2) AS STRING), COUNT(`entrada`),
    'saida', CAST(SUM(`saida`) AS DOUBLE), CAST(ROUND(SUM(`saida`), 2) AS STRING), COUNT(`saida`),
    'estoque_final', CAST(SUM(`estoque_final`) AS DOUBLE), CAST(ROUND(SUM(`estoque_final`), 2) AS STRING), COUNT(`estoque_final`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 11. Datas armazenadas como STRING

**Armadilha:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
Mesma data, formato diferente — normalizar para `AAAAMMDD` antes de comparar.

In [ ]:
-- 11. DATAS EM STRING
SELECT coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo,
       CASE WHEN (CASE WHEN fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato' ELSE 'formato unico' END AS veredito
FROM (
  SELECT stack(1,
    'dt_estoque', COUNT_IF(`dt_estoque` IS NULL OR trim(`dt_estoque`) = ''), COUNT_IF(trim(`dt_estoque`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_estoque`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_estoque`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_estoque`) NOT IN ('', '00000000') THEN `dt_estoque` END), MAX(CASE WHEN trim(`dt_estoque`) NOT IN ('', '00000000') THEN `dt_estoque` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo)
  FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 12. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [ ]:
-- 12. CODIGOS
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(3,
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_empresa', 'string', COUNT_IF(CAST(`cod_empresa` AS STRING) IS NULL OR trim(CAST(`cod_empresa` AS STRING)) = ''), MIN(length(trim(CAST(`cod_empresa` AS STRING)))), MAX(length(trim(CAST(`cod_empresa` AS STRING)))), COUNT_IF(trim(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_empresa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_empresa` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 13. Chaves normalizadas para join com o SAP

Lista das chaves já **sem zeros à esquerda**, prontas para colar no Excel
e cruzar com o extrato do SAP via PROCV/ÍNDICE.

Baixe como CSV e use para identificar registros presentes de um lado e ausentes do outro.

In [ ]:
-- 13. CHAVES NORMALIZADAS (para cruzar com o SAP)
SELECT DISTINCT
       regexp_replace(trim(CAST(`cod_empresa` AS STRING)), '^0+', '') AS `cod_empresa_norm`,
       regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') AS `cod_centro_norm`,
       regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS `cod_material_norm`,
       regexp_replace(trim(CAST(`dt_estoque` AS STRING)), '^0+', '') AS `dt_estoque_norm`
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
ORDER BY 1, 2;

## 14. Checksum de linha

Gera uma impressão digital de cada linha. Dois usos:

- **Contar linhas realmente distintas** — se `linhas` for maior que `linhas_unicas`,
  existem registros 100% idênticos (duplicata real)
- **Comparação rápida** — aplicando a mesma concatenação no SAP, dá para achar
  divergências sem comparar campo a campo

In [ ]:
-- 14. CHECKSUM DE LINHA
SELECT COUNT(*)                                                 AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida` AS STRING), ''), COALESCE(CAST(`dt_estoque` AS STRING), ''), COALESCE(CAST(`estoque_inicial` AS STRING), ''), COALESCE(CAST(`entrada` AS STRING), ''), COALESCE(CAST(`saida` AS STRING), ''), COALESCE(CAST(`estoque_final` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''))))             AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida` AS STRING), ''), COALESCE(CAST(`dt_estoque` AS STRING), ''), COALESCE(CAST(`estoque_inicial` AS STRING), ''), COALESCE(CAST(`entrada` AS STRING), ''), COALESCE(CAST(`saida` AS STRING), ''), COALESCE(CAST(`estoque_final` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''))))  AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida` AS STRING), ''), COALESCE(CAST(`dt_estoque` AS STRING), ''), COALESCE(CAST(`estoque_inicial` AS STRING), ''), COALESCE(CAST(`entrada` AS STRING), ''), COALESCE(CAST(`saida` AS STRING), ''), COALESCE(CAST(`estoque_final` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128';

## 15. Amostra de linhas completas

In [ ]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
ORDER BY `cod_material`, `dt_estoque`
LIMIT 20;

## 16. Distribuição interna do centro 4128

Como o volume se reparte dentro do cenário. Útil para conferir se o extrato do SAP
tem a mesma composição.

In [ ]:
-- 16. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)') AS `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR sg_unidade_medida
SELECT COALESCE(NULLIF(trim(CAST(`sg_unidade_medida` AS STRING)), ''), '(vazio)') AS `sg_unidade_medida`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`sg_unidade_medida` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [ ]:
-- 17. FRESHNESS
-- Tabela sem coluna de data de ingestao. Use o historico de gravacao.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario LIMIT 10;

## 18. Análises específicas — MB5B

### 18.1 Equação contábil
`estoque_inicial + entrada − saída = estoque_final`

In [ ]:
-- 18.1 EQUACAO CONTABIL
SELECT COUNT(*) AS linhas,
       COUNT_IF(ABS((estoque_inicial + entrada - saida) - estoque_final) <= 0.001) AS equacao_ok,
       COUNT_IF(ABS((estoque_inicial + entrada - saida) - estoque_final) > 0.001) AS divergentes,
       MAX(ABS((estoque_inicial + entrada - saida) - estoque_final)) AS maior_diferenca,
       CASE WHEN COUNT_IF(ABS((estoque_inicial + entrada - saida) - estoque_final) > 0.001) = 0
            THEN 'OK - equacao fecha' ELSE 'ERRO - investigar' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128';

In [ ]:
-- 18.1b PIORES DIVERGENCIAS
SELECT cod_empresa, cod_material, dt_estoque,
       estoque_inicial, entrada, saida, estoque_final,
       estoque_inicial + entrada - saida AS calculado,
       ABS((estoque_inicial + entrada - saida) - estoque_final) AS diferenca
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
  AND ABS((estoque_inicial + entrada - saida) - estoque_final) > 0.001
ORDER BY diferenca DESC
LIMIT 20;

### 18.2 Cobertura temporal

In [ ]:
-- 18.2 COBERTURA TEMPORAL
WITH d AS (
  SELECT COALESCE(to_date(CASE WHEN dt_estoque RLIKE '^[0-9]{8}$' THEN dt_estoque END, 'yyyyMMdd'),
                  to_date(dt_estoque)) AS dt
  FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'
)
SELECT MIN(dt) AS primeira_data, MAX(dt) AS ultima_data,
       COUNT(DISTINCT dt) AS dias_com_dado,
       datediff(MAX(dt), MIN(dt)) + 1 AS dias_do_periodo,
       datediff(MAX(dt), MIN(dt)) + 1 - COUNT(DISTINCT dt) AS dias_sem_dado
FROM d;

## 19. EXTRAÇÃO COMPLETA — centro 4128

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for muito grande, use a célula 19.1.

In [ ]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario
WHERE `cod_centro` = '4128'
ORDER BY `cod_material`, `dt_estoque`;

### 19.1 Alternativa para volume grande _(opcional)_

Descomente para gravar o resultado numa tabela própria e exportar de lá sem limite de tela.

In [ ]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_mb5b_4128 AS
-- SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_mb5b_4128;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

Bloco final. **Copie esta saída** e envie ao agente junto com o notebook.

In [ ]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128'),
g AS (
  SELECT 'cod_empresa + cod_centro + cod_material + dt_estoque' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_centro + cod_material + dt_estoque' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_centro`, `cod_material`, `dt_estoque` FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_empresa + cod_centro + cod_material + dt_estoque + sg_unidade_medida' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, `sg_unidade_medida` FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario WHERE `cod_centro` = '4128')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'MB5B' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4128'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '12'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4128 no Datalake.
2. Extrair a mesma transação no SAP com o filtro `centro = 4128`, **todas as abas**.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Zeros à esquerda normalizados nos dois lados (seção 12)
- [ ] Formato de data normalizado (seção 11)
- [ ] Totais numéricos conferidos (seção 10)
- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP (seção 6)
